[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lbutler2405/EMP5027-rows-to-pixels/blob/main/notebooks/practical-4-eda-stats/EMP5027-Lecture-4-EDA-and-Stats.ipynb)



# EMP5027: Methods in Data Analysis & Quality Assurance
## Lecture 4: **Data Visualisation and Exploratory Data Analysis (EDA)**

**Instructor:** Dr. Liam Butler · University of Malta

We use real datasets throughout and put the emphasis on clear, reproducible EDA, the kind of exploratory work you should do before running a statistical test or building a model on environmental monitoring data.



## Learning Objectives
By the end of this practical, you will be able to:
- Perform univariate and bivariate EDA with histograms, KDEs, box and violin plots, FacetGrid, and pairplots.
- Create correlation heatmaps, custom scatter visuals, and visualise missing data.
- Use 3D and interactive plots where they genuinely help, and know when to avoid them.
- Compare multiple time series and build a dual-axis figure, understanding its caveats.
- Run core environmental statistics: t-test, ANOVA with Tukey HSD, and linear and logistic regression.
- Fit a multivariable OLS model, check collinearity with VIF, and select variables with backward elimination and forward selection by AIC.
- Fit GLMs (Gamma, log link) and GAMs for non-linear patterns, and check residuals and model fit.
- Perform hierarchical clustering and interpret the resulting clusters, and test for trends with the Mann–Kendall test.
- Make a simple spatial plot with GeoPandas and add a basemap for geographic context.



---
## Part 0: Setup
We import the libraries we will need throughout the notebook. A few of them (missingno, pygam, pymannkendall, contextily, geopandas) are optional extras used only in specific sections, so we wrap those imports in `try/except` blocks and simply skip the relevant example later if a package is not installed.


## Running this in Google Colab

Click the badge above to open this notebook directly in Colab, no local setup required. This notebook needs a couple of packages that are not preinstalled on Colab, so the cell below installs them automatically.

It installs the optional packages used for the missing-data visuals, the GAM, the Mann–Kendall trend test, and the basemap and spatial demo in Part 4.


In [ ]:
# --- Google Colab setup (safe to run locally too, it just skips this step) ---
import sys

if "google.colab" in sys.modules:
    !pip install -q missingno pygam pymannkendall contextily geopandas

    print("Running in Colab, ready to go.")
else:
    print("Not running in Colab, assuming packages are already installed locally.")

In [ ]:
import os
os.getcwd()

In [ ]:
# Core scientific stack
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Visuals
try:
    import seaborn as sns
except Exception as e:
    raise ImportError("Seaborn is required for several examples. Install with `pip install seaborn`.") from e

# Stats & modelling
from scipy.stats import ttest_ind, f_oneway
import statsmodels.api as sm
import statsmodels.formula.api as smf

# Optional libraries (missing → examples are skipped gracefully)
try:
    import missingno as msno
except Exception as e:
    msno = None

try:
    import plotly.express as px
except Exception as e:
    px = None

try:
    import pymannkendall as mk
except Exception as e:
    mk = None

try:
    from pygam import LinearGAM, s
except Exception as e:
    LinearGAM, s = None, None

# Plot defaults
plt.rcParams["figure.figsize"] = (8, 5)
sns.set_theme(style="whitegrid")



---
# Part 1: EDA with **Penguins** (Univariate & Bivariate)

We will use the penguins dataset to work through the core EDA plot types: distribution plots, box and violin plots, FacetGrid, pairplot, a correlation heatmap, a custom scatter, a missingness visual, and a 3D and interactive plot. Penguins is a good dataset for this because it is small, has a handful of continuous morphological measurements, a few categorical grouping variables (species, island, sex), and some real missing values, so it lets us practise the full EDA workflow on something you can hold in your head.

We load it once here and reuse it for the rest of Part 1.


In [ ]:
# Load penguins ONCE for Part 1
penguins = pd.read_csv("Penguins_data.csv")
penguins_raw = penguins.copy()
penguins

In [ ]:
penguins = penguins.dropna()
print("Penguins shape (after minimal NA drop):", penguins.shape)
penguins.head()


## 1.1 Histogram + KDE (by Group)
A histogram with an overlaid kernel density estimate (KDE) is usually the first plot you should make for any continuous variable. It shows you the shape of the distribution, whether it is skewed or multimodal, and roughly where the centre of the data sits. We colour by species here so we can see immediately whether the three penguin species have different body mass distributions, which is exactly the kind of pattern you are looking for before running any formal test.

Start simple with the basic histogram, and only add overlays like mean and median lines once you have a reason to want them.


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.lines as mlines

# Plot the histogram with hue for species
fig, ax = plt.subplots(figsize=(8,5))
sns.histplot(
    data=penguins,
    x="body_mass_g",
    kde=True,
    hue="species",
    bins=20,
    multiple="stack",
    edgecolor="black",
    ax=ax
)

# Add vertical mean and median lines
ax.axvline(penguins["body_mass_g"].mean(), linestyle="--", color="black", label="Mean") # add mean line
ax.axvline(penguins["body_mass_g"].median(), linestyle=":", color="purple", label="Median") # add median line

ax.grid(False) # remove grid lines
sns.despine(top=True, right=True) #remove top and right border lines

# Change axis (spine) color and thickness
ax.spines['bottom'].set_color('black')
ax.spines['left'].set_color('black')
ax.spines['bottom'].set_linewidth(4.5)
ax.spines['left'].set_linewidth(1.5)

ax.set_title("Distribution of Penguin Body Mass by Species")
ax.set_xlabel("Body Mass (g)")
ax.set_ylabel("Count")

# Build species patches for legend
palette = sns.color_palette(n_colors=penguins["species"].nunique())
species_list = penguins["species"].unique()
species_patches = [mpatches.Patch(color=palette[i], label=species_list[i]) for i in range(len(species_list))]

# Build line legend handles for mean and median
mean_line = mlines.Line2D([], [], color="black", linestyle="--", label="Mean")
median_line = mlines.Line2D([], [], color="purple", linestyle=":", label="Median")

# Combine into one legend
ax.legend(handles=species_patches + [mean_line, median_line], title="Legend", loc="upper right")

plt.tight_layout()
plt.show()


## 1.2 Box + Swarm (or Strip) by Group
A boxplot alone hides the raw data behind a handful of summary statistics. Overlaying the individual points, as a swarm or strip plot, lets you see the actual spread, any gaps, and any outliers that the box would otherwise smooth over. This combination is particularly useful in environmental and ecological work, where sample sizes per group are often small enough that you want to see every point.


In [ ]:
fig, ax = plt.subplots(figsize=(10,5))
sns.boxplot(data=penguins, x="species", y="flipper_length_mm", hue="sex", width=0.6, fliersize=3, ax=ax)
sns.swarmplot(data=penguins, x="species", y="flipper_length_mm", hue="sex",
              dodge=True, alpha=0.6, size=3, linewidth=0.3, ax=ax)

# De-duplicate legend entries
handles, labels = ax.get_legend_handles_labels()

ax.grid(False)

# Change axis (spine) color and thickness
ax.spines['bottom'].set_color('black')
ax.spines['left'].set_color('black')
ax.spines['bottom'].set_linewidth(1.5)
ax.spines['left'].set_linewidth(1.5)

# Place legend outside (right side)
ax.legend(
    handles[:2],
    labels[:2],
    title="Sex",
    bbox_to_anchor=(1.05, 1),   # (x, y) relative to axes
    loc='upper left',           # anchor corner of the legend box
    borderaxespad=0
)

ax.set_title("Flipper Length by Species and Sex")
plt.tight_layout()
plt.show()


## 1.3 Violin Plots (Split by Subgroup)
A violin plot shows the full shape of the distribution rather than just quartiles, which makes it easier to spot multimodality that a boxplot would hide. Setting `split=True` lets us compare two subgroups (here, sex) side by side within each main category, a compact way of checking for an interaction between two grouping variables.


In [ ]:
fig, ax = plt.subplots()
sns.violinplot(data=penguins, x="island", y="body_mass_g", hue="sex", split=True, inner="quartile", ax=ax)
ax.set_title("Body Mass by Island and Sex")
ax.legend(title="Sex", loc="upper right")
plt.tight_layout(); plt.show()


## 1.4 FacetGrid (Small Multiples)
Rather than cramming several groups into one axis with overlapping colours, a FacetGrid gives each group its own small panel on the same scale. This "small multiples" approach is often clearer than a single busy plot, especially once you have more than two or three groups to compare.


In [ ]:
g = sns.FacetGrid(penguins, col="species", height=3.8, aspect=1.2)
g.map_dataframe(sns.histplot, x="bill_length_mm", kde=True, edgecolor="white")
g.set_axis_labels("Bill Length (mm)", "Count")
g.set_titles(col_template="{col_name}")
g.figure.suptitle("Bill Length Distributions by Species", y=1.02)
g.tight_layout(); plt.show()

In [ ]:
## OR to assign different colours to the different plots
import seaborn as sns
import matplotlib.pyplot as plt

# Choose a color palette with as many colors as species
palette = sns.color_palette("Set2", n_colors=penguins["species"].nunique())

# Create the FacetGrid
g = sns.FacetGrid(
    penguins,
    col="species",
    height=3.8,
    aspect=1.2,
    col_wrap=3  # wraps if you have many species: currently this puts each graph in one row with 3 columns (Try changing it to 1)
)

# Map the plots, assigning a different color to each facet
for ax, color, species in zip(g.axes.flat, palette, penguins["species"].unique()):
    sns.histplot(
        data=penguins[penguins["species"] == species],
        x="bill_length_mm",
        kde=True,
        edgecolor="white",
        color=color,
        ax=ax
    )

# Add titles and labels
g.set_axis_labels("Bill Length (mm)", "Count")
g.set_titles(col_template="{col_name}")
g.figure.suptitle("Bill Length Distributions by Species", y=1.02)
g.tight_layout()
plt.show()


## 1.5 Pairplot (Quick Multivariate View)
A pairplot gives you every pairwise scatter plot between a set of continuous variables in one grid, with histograms on the diagonal. It is a fast way to scan for relationships and separation between groups across several variables at once, before deciding which pairs are worth a closer look. Colouring by species here shows how well the morphological measurements separate the three species.


In [ ]:
vars_to_plot = ["bill_length_mm","bill_depth_mm","flipper_length_mm","body_mass_g"]
sns.pairplot(penguins, vars=vars_to_plot, hue="species", corner=True)
plt.show()


## 1.6 Correlation Heatmap
A correlation heatmap summarises the pairwise linear relationships between all your numeric variables at a glance, which is useful for spotting redundant variables before modelling. Since the matrix is symmetric, we mask the upper triangle so we are not showing the same values twice.


In [ ]:
corr = penguins.corr(numeric_only=True)
mask = np.triu(np.ones_like(corr, dtype=bool))
fig, ax = plt.subplots()
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", vmin=-1, vmax=1, cmap="coolwarm", square=True, cbar_kws={"shrink":0.8}, ax=ax)
ax.set_title("Correlation Matrix: Penguin Morphology")
plt.tight_layout(); plt.show()


## 1.7 Custom Scatter (Matplotlib control)
Seaborn is fast for exploration, but sometimes you want full control over markers, colours, and the legend, for example when preparing a figure for a report or a publication. Building the scatter plot directly in matplotlib gives you that control, at the cost of a little more code.


In [ ]:
# Simple custom scatter (per species/sex loop)
species_list = sorted(penguins['species'].unique())
sex_list = sorted(penguins['sex'].dropna().unique())

fig, ax = plt.subplots()
for (sp, sx), grp in penguins.groupby(['species','sex']):
    ax.scatter(grp['flipper_length_mm'], grp['body_mass_g'], label=f"{sp} - {sx}", alpha=0.7, edgecolors="black", s=50)
ax.set_title("Flipper Length vs Body Mass")
ax.set_xlabel("Flipper Length (mm)")
ax.set_ylabel("Body Mass (g)")
ax.grid(True, linestyle="--", alpha=0.5)
ax.legend(fontsize=8, title="Species · Sex")
plt.tight_layout(); plt.show()

In [ ]:
## OR if we want to add trend lines for each species:

import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# Species list + consistent colors
species_list = sorted(penguins['species'].dropna().unique())
palette = sns.color_palette("Set2", n_colors=len(species_list))
species_colors = dict(zip(species_list, palette))

fig, ax = plt.subplots(figsize=(7,5))

# Scatter: keep your species·sex points (any color scheme you like)
for (sp, sx), grp in penguins.groupby(['species', 'sex']):
    ax.scatter(
        grp['flipper_length_mm'], grp['body_mass_g'],
        label=f"{sp}{sx}", alpha=0.7, edgecolors="black", s=50
    )

# Trend lines: one per species, same color + CI
for sp in species_list:
    sub = penguins.loc[penguins['species'] == sp, ['flipper_length_mm','body_mass_g']].dropna()
    if len(sub) > 2:  # need enough points to fit
        sns.regplot( # Plots data and fits a linear model
            data=sub,
            x='flipper_length_mm', y='body_mass_g',
            ax=ax,
            scatter=False,           # don't add extra points
            ci=95,                   # show 95% CI
            color=species_colors[sp],
            line_kws={'linewidth': 2}
        )

ax.set_title("Flipper Length vs Body Mass")
ax.set_xlabel("Flipper Length (mm)")
ax.set_ylabel("Body Mass (g)")
ax.grid(True, linestyle="--", alpha=0.5)
ax.legend(fontsize=8, title="Species and Sex")
plt.tight_layout()
plt.show()


## 1.8 Visualising Missing Data (Optional: `missingno`)
Before you drop or impute missing values, it helps to see where they are. The missingno package gives two useful views: a matrix showing exactly which rows and columns have gaps, and a heatmap showing whether missingness in one variable tends to co-occur with missingness in another. That second view matters ecologically: if, say, bill measurements and body mass tend to go missing together, that points to a shared cause, such as a damaged specimen or a skipped measurement session, rather than values missing at random, and it affects how safely you can drop or impute them.


In [ ]:
if msno is not None:
    msno.matrix(penguins_raw, figsize=(8,4), sparkline=False)
    plt.show()
    msno.heatmap(penguins_raw, figsize=(6,5))
    plt.show()
else:
    print("missingno not installed, skipping missingness visuals. Install with `pip install missingno`.")


## 1.9 3D & Interactive (Use Sparingly)
A 3D scatter can reveal structure among three continuous variables that you would miss looking at them two at a time, but 3D plots are hard to read from a static image because you lose the ability to rotate the view. An interactive plot, here with Plotly, supports zoom, pan, and hover tooltips, which is excellent for exploring your own data or for a live demo, but is less useful once you need a fixed figure for a report. Use both sparingly, and prefer simpler 2D plots when they tell the same story.


In [ ]:
# 3D scatter (requires mpl_toolkits)
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
fig = plt.figure(figsize=(8,6))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(penguins['bill_length_mm'], penguins['flipper_length_mm'], penguins['body_mass_g'], alpha=0.8, edgecolor="k", s=40)
ax.set_xlabel("Bill Length (mm)"); ax.set_ylabel("Flipper (mm)"); ax.set_zlabel("Body Mass (g)")
ax.set_title("3D Morphospace: Penguins")
plt.show()

In [ ]:
import plotly.io as pio
from plotly.offline import init_notebook_mode
import plotly.io as pio
import plotly.express as px

# Try this:
#pio.renderers.default = "notebook_connected"   # tries CDN

## OR the below:

init_notebook_mode(connected=False)   # embed JS inline
pio.renderers.default = "notebook"    # use inline renderer

fig = px.scatter(
    penguins,
    x="flipper_length_mm",
    y="body_mass_g",
    color="species",
    symbol="sex",
    hover_data=["island", "bill_length_mm", "bill_depth_mm"],
    title="Interactive: Body Mass vs Flipper Length"
)
#plt.savefig(dpi = 300)
fig.show()


---
# Part 2: **Multiple Time Series** (Air Quality NO₂)

We move now from single-snapshot morphological data to a time series problem. We will compare NO2 concentrations across several monitoring locations, inspect gaps in the record, and build a dual-axis plot, which we use as an example of a chart type to handle with caution rather than as a default choice.


In [ ]:
# Load ONCE for Part 2
air = pd.read_csv("air_quality_data_2.csv", parse_dates=["date.utc"])
print("Air quality rows:", len(air))
air.head()


## 2.1 Multi-location Time Series
Plotting each location as its own line on a shared time axis is the natural first plot for this kind of data. It lets you compare both the overall level of NO2 at each site and how the sites move together over time, and it is usually the quickest way to spot missing periods, sudden spikes, or a location that behaves very differently from the rest.


In [ ]:
fig, ax = plt.subplots(figsize=(10,6))
for loc, grp in air.groupby("location"):
    ax.plot(grp["date.utc"], grp["value"], label=loc)
ax.set_title("$NO_2$ Concentration Over Time: Multiple Cities")
ax.set_xlabel("Date"); ax.set_ylabel("$NO_2$ (µg/m³)")
ax.legend(title="Location")
ax.grid(True, linestyle="--", alpha=0.4)
plt.xticks(rotation=45); plt.tight_layout(); plt.show()


## 2.2 Dual-Axis Plot (Use Carefully)
A dual-axis plot puts two variables with different scales or units on the same figure, each with its own y-axis. It can be convenient, but it is also one of the easier chart types to use to mislead a reader, deliberately or not, because the two axes can be scaled to make two unrelated series look like they move together. Use it only when the variables genuinely belong on the same plot, and always label both axes clearly so the reader can tell which line belongs to which axis.


In [ ]:
# London vs Paris NO₂ on twin axes (illustrative only)
london = air[air["location"] == "London Westminster"]
paris  = air[air["location"] == "Paris"]

fig, ax1 = plt.subplots(figsize=(10,6))
ax2 = ax1.twinx()
ax1.plot(london["date.utc"], london["value"], label="London")
ax2.plot(paris["date.utc"], paris["value"], label="Paris")

ax1.set_ylabel(r"London $NO_2$ (µg/m³)")
ax2.set_ylabel(r"Paris $NO_2$ (µg/m³)")
ax1.set_xlabel("Date")
ax1.grid(True, linestyle="--", alpha=0.5)
plt.title(r"$NO_2$ Trends: London vs Paris (Dual Axes)")

fig.tight_layout(); plt.show()


---
# Part 3: **Environmental Statistics** with Water Quality

We now move to a substantial environmental dataset: the Kaggle **Water Quality** dataset (download it beforehand and place it alongside this notebook). This is where we get into the statistics proper: t-tests, ANOVA with Tukey HSD, correlation heatmaps, linear and logistic regression, multivariable OLS with VIF and variable selection, a GLM (Gamma, log link), a GAM, hierarchical clustering, and the Mann–Kendall trend test. Each technique is introduced against a specific, realistic question about the water quality data, rather than in the abstract.



### Load & Inspect
Place `BKB_WaterQualityData_2020084.csv` in the same folder as this notebook, or point the path below at wherever you saved it. As is typical of real-world data, the column names are inconsistent, with extra whitespace, stray line breaks, and mixed units in the header, so before doing anything else we standardise the column names, then take a first look at the table.


In [ ]:
# Load ONCE for Part 3
wq = pd.read_csv("BKB_WaterQualityData_2020084.csv")
wq.columns = wq.columns.str.strip().str.replace("\n", " ").str.replace("\r", " ").str.replace("  ", " ")
wq.head()

In [ ]:
wq.columns

In [ ]:
# Rename for easier typing
rename_map = {
    "Dissolved Oxygen (mg/L)": "DO_mg_L",
    "Salinity (ppt)": "Salinity_ppt",
    "BOD (mg/L)": "BOD_mg_L",
    "pH (standard units)": "pH",
    "Salinity (ppt)": "Salinity_ppt",
    "Water Temp (?C)": "WaterTemp_C",  # alternate encoding fallback
    "Air Temp (?F)": "Air_Temp_F",
    "State": "State",
    "read_date": "read_date",
    "AirTemp (C)": "Air_Temp_C"
}
wq = wq.rename(columns={k:v for k,v in rename_map.items() if k in wq.columns})
wq.info()

In [ ]:
wq

In [ ]:
wq_drop = wq.drop(columns=['Field_Tech', 'DateVerified', 'WhoVerified', 'Air Temp-Celsius'])
display(wq_drop)


### Distributions (Histogram Grid)
Before choosing a statistical test or a model, it is worth looking at the shape of each variable you plan to use. Skewed or heavy-tailed distributions may need a transformation, or point you towards a non-parametric test or a GLM with a non-Gaussian family, rather than an ordinary linear model. We plot four key water quality variables side by side here so we can compare their shapes in one go.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Select key water-quality variables
cols = ["DO_mg_L", "pH", "Salinity_ppt", "WaterTemp_C"]
palette = sns.color_palette("Set2", len(cols))

# Create the subplots manually
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()

# Plot each histogram separately so we can control color, style, etc.
for ax, col, color in zip(axes, cols, palette):
    ax.hist(wq[col].dropna(), bins=30, edgecolor="black", color=color)
    ax.set_title(col.replace("_", " "), fontsize=11)
    ax.set_xlabel("Measured Value", fontsize=10)
    ax.set_ylabel("Frequency", fontsize=10)

    # Clean up axes
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["bottom"].set_color("black")
    ax.spines["left"].set_color("black")
    ax.spines["bottom"].set_linewidth(1.2)
    ax.spines["left"].set_linewidth(1.2)

    # Add a subtle grid for readability
    ax.grid(True, linestyle="--", linewidth=0.5, alpha=0.5)

# Global title and layout
fig.suptitle("Distribution of Key Water Quality Indicators", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()


## 3.1 Two-Sample t-test
The t-test compares the means of a continuous variable between two groups. We use Welch's version (`equal_var=False`) rather than the classic Student's t-test, because Welch's test does not assume the two groups have equal variance, a safer default for real environmental data where variability often genuinely differs between sites.


In [ ]:
wq_drop

In [ ]:
print(wq_drop['Site_Id'].unique())

In [ ]:
from scipy.stats import ttest_ind

# Select just the two states and the variable of interest
site_bay = wq_drop[wq_drop["Site_Id"]== 'Bay']
site_A = wq_drop[wq_drop["Site_Id"]== 'A']
display(site_bay), display(site_A)

In [ ]:
print("Missing Values for Bay:\n", site_bay.isna().sum())
print("\n")
print("Missing Values for A:\n", site_A.isna().sum())

In [ ]:
# Drop missing values
site_bay = site_bay.dropna(subset=["Secchi Depth (m)"])
site_A = site_A.dropna(subset=["Secchi Depth (m)"])
display(site_bay)
display(site_A)

In [ ]:
# Run independent t-test
t_stat, p_val = ttest_ind(site_bay['Secchi Depth (m)'], site_A['Secchi Depth (m)'], equal_var=False)

print(f"T-statistic: {t_stat:.2f}")
print(f"P-value: {p_val:.4f}")


## 3.2 ANOVA + Tukey HSD
A one-way ANOVA extends the two-sample t-test to more than two groups, and tests whether at least one group mean differs from the others. It does not, on its own, tell you which groups differ. For that we follow it with a Tukey HSD post-hoc test, which compares every pair of groups while controlling for the fact that we are now making many comparisons at once.


In [ ]:
import pandas as pd
from scipy.stats import f_oneway
from statsmodels.stats.multicomp import pairwise_tukeyhsd
import seaborn as sns
import matplotlib.pyplot as plt

# Keep just what we need and clean labels
df = wq[['Site_Id', 'WaterTemp_C']].dropna().copy()
df['Site_Id'] = df['Site_Id'].astype(str).str.strip().str.upper()   # merges 'D' and 'd'
df = df[df['Site_Id'].isin(['BAY','A','B','C','D'])]

# see how many rows per site
print(df['Site_Id'].value_counts())

In [ ]:
# Run one-way ANOVA
groups = [grp['WaterTemp_C'] for _, grp in df.groupby('Site_Id')]
F, p = f_oneway(*groups)
print(f"ANOVA: F = {F:.2f}, p = {p:.4f}\n")

In [ ]:
# Run Tukey HSD post-hoc test
tukey = pairwise_tukeyhsd(endog=df['WaterTemp_C'], groups=df['Site_Id'], alpha=0.05)
print(tukey)


## 3.3 Logistic Regression (Binary outcome, e.g., Water Temp > 18)
Logistic regression is the natural tool when your outcome is binary rather than continuous, for example whether a reading counts as "warm water" or not against some threshold. This kind of binary alert is exactly the sort of output an environmental monitoring system might need to flag automatically. We use scikit-learn here, which is convenient for the train/test split and evaluation metrics, in contrast to the statsmodels-based regressions used elsewhere in this notebook.


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

# Pick columns that exist and make names tidy
df_log = wq_drop[['WaterTemp_C', 'Air_Temp_C', 'Salinity_ppt', 'DO_mg_L', 'Water Depth (m)']].rename(
    columns={'Water Depth (m)': 'WaterDepth_m'}
).dropna()
display(df_log.head())

In [ ]:
# Target: "Warm water" (adjust threshold if you like)
df_log['WarmWater'] = (df_log['WaterTemp_C'] > 18).astype(int)
df_log

In [ ]:
# 3) Features and split
X = df_log[['Air_Temp_C', 'Salinity_ppt', 'DO_mg_L', 'WaterDepth_m']]
y = df_log['WarmWater']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
display(X), display(y)

In [ ]:
print(X_train.shape), print(X_test.shape), print(y_train.shape), print(y_test.shape)

In [ ]:
# Fit + evaluate
clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)
print(classification_report(y_test, y_pred))

In [ ]:
# Probability plot (how confident the model is)
probs = clf.predict_proba(X_test)[:, 1]
plt.hist(probs, bins=30, edgecolor='black')
plt.title('Predicted Probability of Warm Water (>18°C)')
plt.xlabel('Probability'); plt.ylabel('Count')
plt.tight_layout(); plt.show()

In [ ]:
probs

In [ ]:
# Show feature effects
coef = pd.Series(clf.coef_[0], index=X.columns).sort_values()
print("Logistic regression coefficients:\n", coef)

# Quick confusion matrix
ConfusionMatrixDisplay.from_estimator(clf, X_test, y_test, colorbar=False)
plt.title('Confusion Matrix (test set)')
plt.tight_layout(); plt.show()


## 3.4 Multivariable Linear Regression + VIF
A single-predictor regression can be misleading if that predictor is correlated with other variables that also affect the outcome. Multivariable regression lets us control for several predictors at once, so each coefficient reflects the effect of that variable while holding the others constant, at least in principle. Because the predictors can themselves be correlated with each other, we also check the Variance Inflation Factor (VIF) for each one: a high VIF means a predictor's information is largely duplicated by the others, which makes its individual coefficient unstable and hard to interpret.


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

# Tidy names (spaces/parentheses → underscores) for easier modeling
lin_vif = wq_drop.rename(columns={
    "Water Depth (m)": "WaterDepth_m",
    "Secchi Depth (m)": "SecchiDepth_m"
})

# Choose target + predictors that exist in your table
target = "DO_mg_L"
features = ["Air_Temp_C", "WaterTemp_C", "Salinity_ppt", "pH", "WaterDepth_m", "SecchiDepth_m"]

# Keep only needed columns and drop missing rows (simple, explicit for teaching)
dfm = lin_vif[[target] + features].dropna().copy()
display(dfm)

In [ ]:
# Build design matrix X and response y
X = sm.add_constant(dfm[features])   # adds intercept term
y = dfm[target]
print(X), print(y)

In [ ]:
# Fit OLS and print a readable summary
ols_model = sm.OLS(y, X).fit()
print(ols_model.summary()) ## many different parameters

# Interpreting the OLS Regression Output

This section explains each part of the **OLS (Ordinary Least Squares)** regression summary from our model predicting **Dissolved Oxygen (DO_mg_L)**.

---

## Top Section: Model Information

| Term | Meaning |
|------|----------|
| **Dep. Variable** | The variable we are trying to predict: here it is `DO_mg_L` (Dissolved Oxygen). |
| **Model** | The regression type. `OLS` stands for *Ordinary Least Squares*. |
| **Method** | The estimation method (minimises the sum of squared residuals). |
| **Date / Time** | When the model was run (for record-keeping). |
| **No. Observations** | Number of valid rows used after removing missing values (`1320`). |
| **Df Residuals** | Degrees of freedom for residuals, observations minus parameters. |
| **Df Model** | Number of predictors (independent variables) in the model (`6`). |
| **Covariance Type** | How the standard errors were estimated (default: non-robust). |

---

## Model Fit Statistics

| Statistic | Meaning |
|------------|----------|
| **R-squared = 0.400** | About **40% of the variation** in Dissolved Oxygen is explained by our predictors. |
| **Adj. R-squared = 0.397** | Adjusted R² (accounts for the number of predictors). Slightly lower, as expected. |
| **F-statistic = 145.8** | Tests whether the model, overall, explains a significant amount of variation. |
| **Prob (F-statistic) = 9.04e-142** | Very small p-value, so the model as a whole is statistically significant. |
| **Log-Likelihood = −2755.7** | Higher (less negative) values indicate a better fit. |
| **AIC / BIC = 5525 / 5562** | Information criteria used for model comparison. Lower values are better. |

---

## Coefficient Table

Each predictor's effect on `DO_mg_L` is shown below.

| Column | Meaning |
|---------|----------|
| **coef** | Estimated regression coefficient: the change in DO (mg/L) for a one-unit increase in that predictor, keeping others constant. |
| **std err** | Standard error of the estimate. Smaller means more precise. |
| **t** | t-statistic, the coefficient divided by its standard error. |
| **P>|t|** | p-value. If less than 0.05, the predictor is statistically significant. |
| **[0.025, 0.975]** | 95% confidence interval, the range of plausible coefficient values. |

---

### Interpretation of Each Predictor

| Variable | Coefficient | Meaning |
|-----------|--------------|----------|
| **const = 7.16** | Baseline DO when all predictors are 0. |
| **Air_Temp_C = 0.0024** | Not significant (p = 0.724): no clear effect on DO after controlling for other variables. |
| **WaterTemp_C = −0.1691** | Strong, negative, and significant. For every 1°C increase in water temperature, DO decreases by about 0.17 mg/L. |
| **Salinity_ppt = +0.5462** | Positive and significant. Higher salinity corresponds to slightly higher DO (could be site-related). |
| **pH = +0.2618** | Positive and significant. Higher pH is associated with slightly higher DO. |
| **WaterDepth_m = +0.7260** | Positive and significant. Deeper waters tend to have higher DO. |
| **SecchiDepth_m = −0.7093** | Negative and significant. Clearer water (higher Secchi depth) slightly decreases DO, perhaps due to less surface mixing. |

---

## Model Diagnostics

| Term | Meaning |
|------|----------|
| **Omnibus / Prob(Omnibus)** | Tests whether residuals are normally distributed. p = 0 means residuals are **not perfectly normal**. |
| **Skew / Kurtosis** | Shape of residual distribution (ideal: Skew ≈ 0, Kurtosis ≈ 3). |
| **Durbin–Watson = 1.19** | Checks for autocorrelation in residuals (≈2 means no autocorrelation). Here, slightly low, so there may be some positive correlation. |
| **Jarque–Bera (JB) / Prob(JB)** | Another test for residual normality. A very small p confirms non-normality. |
| **Cond. No. = 267** | Condition number. Large values (>30) suggest multicollinearity. Here it is moderate: some correlation among predictors, but not extreme. |

---

## Summary

| Concept | Example interpretation |
|----------|------------------------|
| **Significant (p < 0.05)** | Predictor likely has a real effect on DO. |
| **Negative coefficient** | As the variable increases, DO tends to decrease (e.g., `WaterTemp_C`). |
| **Positive coefficient** | As the variable increases, DO tends to increase (e.g., `Depth`, `pH`). |
| **R² = 0.4** | Model explains 40% of DO variation, which is realistic for environmental data. |
| **Residuals** | Should look like a random scatter around 0 (check with residual plot). |

---

### Main concepts
- **Water temperature** is the strongest predictor of DO, and the relationship is inverse.
- **Salinity, pH, and depth** also matter, but to a lesser degree.
- **Model assumptions** (normal residuals, no strong multicollinearity) should always be checked.
- An R² of 0.4 indicates a *moderately strong* model for natural environmental data.

---


In [ ]:
# Variance Inflation Factor (VIF): diagnose collinearity among predictors
#    (skip the constant when reporting VIF)
vif = pd.DataFrame({
    "feature": X.columns[1:],  # exclude 'const'
    "VIF": [variance_inflation_factor(X.values, i) for i in range(1, X.shape[1])]
}).sort_values("VIF", ascending=False)
print("\nVariance Inflation Factors:\n", vif)

In [ ]:
# Simple residuals diagnostics: residuals vs fitted
fig, ax = plt.subplots(figsize=(6,4))
ax.scatter(ols_model.fittedvalues, ols_model.resid, alpha=0.6, edgecolor="black")
ax.axhline(0, color="black", linestyle="--", linewidth=1)
ax.set_xlabel("Fitted DO (mg/L)")
ax.set_ylabel("Residuals")
ax.set_title("Residuals vs Fitted: Multivariable OLS")
sns.despine()
plt.tight_layout()
plt.show()


## 3.4b Variable Selection: Backward Elimination & Forward Selection (AIC)

The regression in 3.4 used six predictors chosen more or less by hand. In practice you often want a more principled way to decide which predictors actually belong in the model, especially once you have several candidates that might be correlated with each other. We will look at two classic, complementary approaches, both built on the `dfm`, `target`, and `features` set up in 3.4:
- **Backward elimination** starts with every candidate predictor in the model and, at each step, drops the one that is least statistically significant. It favours a model you can interpret cleanly, because everything left in it clears a significance threshold.
- **Forward selection (AIC)** starts with no predictors at all and, at each step, adds whichever predictor most improves the Akaike Information Criterion (AIC). It favours predictive parsimony: the smallest model that still explains the data well, judged by a criterion that penalises unnecessary complexity rather than by significance testing.

Neither method is automatically "correct". They can select different predictor sets, and comparing what each one keeps is often more informative than either result on its own.


In [ ]:
# Reuse dfm/target/features from 3.4 above
candidates = features.copy()



### Backward elimination (p-values)
Start with **all** candidate predictors in the model. At each step, fit the model, look at the p-value of every predictor, and remove whichever one has the **highest p-value**, as long as it is above 0.05. Refit with the remaining predictors and repeat. Stop once every predictor left in the model is significant at the 0.05 level. This is a step-down procedure: the model only ever gets smaller, one variable at a time.


In [ ]:
current = candidates.copy()

while True:
    # Fit a model on the predictors still in "current"
    X_be = sm.add_constant(dfm[current])
    model_be = sm.OLS(dfm[target], X_be).fit()
    pvals = model_be.pvalues.drop("const")  # ignore the intercept's p-value
    worst_p = pvals.max()
    if worst_p <= 0.05:
        break  # every remaining predictor is significant, stop here
    worst_var = pvals.idxmax()
    print(f"Removing {worst_var} (p = {worst_p:.4f})")
    current.remove(worst_var)  # drop the least significant predictor and refit

print("\nSelected predictors (backward):", current)
X_final = sm.add_constant(dfm[current])
model_backward = sm.OLS(dfm[target], X_final).fit()
print(model_backward.summary())



### Forward selection (AIC)
Start from an intercept-only model, with **no predictors** at all. At each step, try adding each remaining candidate predictor one at a time, fit that model, and record its AIC. Keep whichever addition gives the **lowest AIC**, but only if that AIC actually improves on the current best. Repeat, adding one predictor per step, and stop as soon as no remaining predictor lowers the AIC any further. Unlike backward elimination, this procedure only ever adds variables, and it is guided by model fit (AIC) rather than by individual p-values.


In [ ]:
remaining = candidates.copy()
selected = []
best_aic = float("inf")
final_model = None

while remaining:
    tried = []
    for var in remaining:
        # Try adding each remaining candidate one at a time and record its AIC
        X_try = sm.add_constant(dfm[selected + [var]])
        m = sm.OLS(dfm[target], X_try).fit()
        tried.append((m.aic, var, m))
    tried.sort()  # sort candidates by AIC (ascending), lowest first
    aic, best_var, m_best = tried[0]
    if aic < best_aic:
        # This addition improves on the current best model, so keep it and continue
        selected.append(best_var)
        remaining.remove(best_var)
        best_aic = aic
        final_model = m_best
        print(f"Added {best_var} | AIC = {best_aic:.2f}")
    else:
        break  # no remaining predictor lowers AIC any further, stop here

print("\nSelected predictors (forward):", selected)
print(final_model.summary())



### Compare & sanity-check
Backward elimination emphasises significance, which suits a model you want to interpret. Forward selection by AIC emphasises parsimony, which suits a model you want to use for prediction. It is entirely possible for the two procedures to land on different predictor sets, and that disagreement is itself useful information about how much the data actually constrains the "right" model.

Whichever model you end up trusting, always run a quick residual check on it before using it for anything, exactly as we did for the full model in 3.4.


In [ ]:
# Residual plot for the forward-selected model
resid = final_model.resid
fitted = final_model.fittedvalues

plt.figure()
plt.scatter(fitted, resid, alpha=0.6)
plt.axhline(0, linestyle="--")
plt.xlabel(f"Fitted {target}")
plt.ylabel("Residuals")
plt.title("Residuals vs Fitted: Forward-Selected OLS")
plt.grid(True)
plt.show()



### Optional next steps
- Refit the final model(s) on a **train** subset and evaluate performance on a **hold-out** set.
- Try a **regularised** approach (e.g. `LassoCV`) for a more stable, automated form of variable selection.
- Recheck **VIF** on the selected predictors below.


In [ ]:
Xv = sm.add_constant(dfm[selected])
vif_selected = pd.DataFrame({
    "feature": Xv.columns,
    "VIF": [variance_inflation_factor(Xv.values, i) for i in range(Xv.shape[1])]
})
vif_selected



## 3.5 GLM (Gamma, log link): Positive, Skewed Response
Dissolved oxygen, like many environmental measurements, is strictly positive and often right-skewed rather than normally distributed. An ordinary linear model does not know that DO cannot go below zero, and it can happily predict negative values for some combinations of predictors. A Generalised Linear Model (GLM) with a Gamma family and a log link is built for exactly this situation: the Gamma family suits a positive, skewed continuous response, and the log link guarantees predictions on the original scale stay positive, since it models the log of the mean as a linear function of the predictors.


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf

# Keep only columns we need and clean rows
df_glm = wq_drop[['DO_mg_L', 'WaterTemp_C', 'Salinity_ppt', 'pH']].dropna().copy()

# Gamma with log link requires y > 0
df_glm = df_glm[df_glm['DO_mg_L'] > 0]
display(df_glm)

In [ ]:
# Fit a simple GLM: DO ~ WaterTemp + Salinity + pH
glm = smf.glm(
    formula='DO_mg_L ~ WaterTemp_C + Salinity_ppt + pH',
    data=df_glm,
    family=sm.families.Gamma(link=sm.families.links.log())
).fit()

print(glm.summary())

In [ ]:
df_glm

In [ ]:
# 3) Predicted vs Actual (with 45° line)
df_glm['DO_pred'] = glm.predict(df_glm)

plt.figure(figsize=(5.5,4))
plt.scatter(df_glm['DO_pred'], df_glm['DO_mg_L'], alpha=0.6, edgecolor='black')
mx = max(df_glm['DO_mg_L'].max(), df_glm['DO_pred'].max())
plt.plot([0, mx], [0, mx], linestyle='--', color='black')
plt.xlabel('Predicted DO (mg/L)')
plt.ylabel('Actual DO (mg/L)')
plt.title('Predicted vs Actual: GLM (Gamma, log link)')
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
# Residuals vs Fitted (deviance residuals)
plt.figure(figsize=(5.5,4))
plt.scatter(glm.fittedvalues, glm.resid_deviance, alpha=0.6, edgecolor='black')
plt.axhline(0, linestyle='--', color='black')
plt.xlabel('Fitted DO (mg/L)')
plt.ylabel('Deviance Residuals')
plt.title('GLM Diagnostics: Residuals vs Fitted')
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

## Residuals Explained
Residuals represent the difference between what the model predicts and what is actually observed.

$\text{Residual} = \text{Observed value} - \text{Predicted value}$

So:
- If the model underestimates an observation, the residual is positive
- If the model overestimates an observation, the residual is negative

You can call residuals the model's "errors", not because the data are wrong, but because the model can never be perfect.

### In our Plot:
- The x-axis is the fitted (predicted) Dissolved Oxygen (DO) from the GLM.
- The y-axis is the deviance residuals, which measure how far each observation is from the model's prediction, adjusted for the Gamma distribution.
- The dashed line at zero is the ideal "no error" reference line.

Each dot is one observation.

The vertical position shows how wrong the model was for that point.

If the model fits well, the dots should be randomly scattered around zero, with no patterns, no curves, and no widening funnel.



## 3.6 GAM (Non-linear Smoother)
Every regression model we have fitted so far assumes a straight-line relationship between each predictor and the response. That is a strong assumption, and it is often wrong for environmental relationships, where a variable's effect can level off, reverse, or otherwise curve. A Generalised Additive Model (GAM) relaxes that assumption: instead of fitting a single coefficient per predictor, it fits a smooth, flexible curve, built from splines, for each predictor, and lets the data decide the shape of that curve rather than forcing it to be linear. We will fit a GAM for Dissolved Oxygen as a function of Water Temperature and see whether the relationship really is a straight line or not.


In [ ]:
wq_drop

In [ ]:
# pip install pygam  # run this once if pygam is not installed

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pygam import LinearGAM, s

# Define x (WaterTemp_C) and y (DO_mg_L) directly from wq_drop
X = wq_drop[["WaterTemp_C"]].dropna().values
y = wq_drop.loc[wq_drop["WaterTemp_C"].notna(), "DO_mg_L"].values
print(X), print(y)


In [ ]:
# Remove any missing DO values
mask = ~np.isnan(y)
X = X[mask]
y = y[mask]
print(X), print(y)

In [ ]:
# Fit a simple GAM: DO ~ s(WaterTemp)
gam = LinearGAM(s(0)).fit(X, y)

# Generate an evenly spaced grid of WaterTemp values and predict DO along it, to draw a smooth curve
xx = np.linspace(X.min(), X.max(), 200).reshape(-1, 1)
yy = gam.predict(xx)

print(xx), print(yy)


This line fits a model that does not force a straight-line relationship between Water Temperature and Dissolved Oxygen.

Instead, it fits a smooth curve, built from splines, to capture however DO actually changes with Water Temperature. The `.fit(X, y)` call tells the model to learn that curve from the data.

`s(0)` tells the model to apply a smoother, a flexible curve rather than a straight line, to the first column of `X`. Here that is `WaterTemp_C`. The number inside the brackets is just the column index (0 for the first column, 1 for the second, and so on), so `s(0)` means fit a smooth term on the first predictor.


In [ ]:
# Plot data + GAM fit
plt.figure(figsize=(7,5))
plt.scatter(X, y, alpha=0.3, s=25, edgecolor="black", label="Data")
plt.plot(xx, yy, color="black", linewidth=2, label="GAM Smooth Fit")
plt.title("GAM: Dissolved Oxygen vs Water Temperature")
plt.xlabel("Water Temperature (°C)")
plt.ylabel("Dissolved Oxygen (mg/L)")
plt.grid(True, linestyle="--", alpha=0.5)
sns.despine()
plt.legend()
plt.tight_layout()
plt.show()


## 3.7 Mann–Kendall Trend Test (Non-parametric)
Many environmental time series are not well described by a straight line, and the Mann–Kendall test is built for exactly that. It asks only whether a series is consistently going up or consistently going down over time, without assuming linearity or a particular distribution for the data. That makes it robust to outliers and to the kind of non-normal, noisy pattern typical of environmental monitoring data, where a standard linear trend test could be misled by a handful of extreme readings.


In [ ]:
# pip install pymannkendall   # ← run once if not already installed

import pandas as pd
import matplotlib.pyplot as plt
import pymannkendall as mk

# Make a copy so we don't change the original data
manny = wq_drop.copy()

# Ensure dates are in datetime format and sorted chronologically
manny["Read_Date"] = pd.to_datetime(manny["Read_Date"], errors="coerce")
manny = manny.sort_values("Read_Date")

# Drop only rows where the specific variable of interest (DO_mg_L) is missing
#     (we're not dropping entire columns, just skipping rows without DO values)
manny = manny[manny["DO_mg_L"].notna()]
display(manny)

In [ ]:
manny.dtypes

In [ ]:
# Run the Mann–Kendall trend test on the Dissolved Oxygen series
#     This is a non-parametric test for monotonic trends over time.
res = mk.original_test(manny["DO_mg_L"])

# Print easy-to-read output
print(f"Trend direction: {res.trend}")
print(f"Significant trend (h): {res.h}")
print(f"p-value: {res.p:.4f}")
print(f"Kendall’s Tau: {res.Tau:.3f}")
print(f"Sen’s slope: {res.slope:.4f}")

In [ ]:
# Plot the DO values over time to visualise the trend
plt.figure(figsize=(8,4))
plt.plot(manny["Read_Date"], manny["DO_mg_L"], color="steelblue", alpha=0.7, label="DO (mg/L)")
plt.title("Dissolved Oxygen Over Time")
plt.xlabel("Date")
plt.ylabel("DO (mg/L)")
plt.grid(True, linestyle="--", alpha=0.4)
plt.legend()
plt.tight_layout()
plt.show()

Mann–Kendall looks for a consistent upward or downward pattern in Dissolved Oxygen over time, without assuming linearity or normality.

This is ideal for environmental data.

The output of the test gives us:
- Trend: "increasing", "decreasing", or "no trend" (direction of change over time).
- h: whether the trend is statistically significant at the default alpha of 0.05. A value of 1 means significant, 0 means not significant.
- p: the p-value for the trend. A smaller value is stronger evidence of a real trend.
- Tau: Kendall's Tau, ranging from −1 to +1. Its sign shows the direction of the trend and its magnitude shows the strength.
- Sen's slope: the estimated median rate of change per time step (e.g., mg/L per observation interval).


In [ ]:
# When to use other variants, e.g. Seasonality (e.g., monthly data):
# The seasonal test splits the series into a fixed number of periods (here 12, for a monthly cycle)
# and tests for a trend within each period, so a repeating seasonal pattern is not mistaken for a trend.
res_seasonal = mk.seasonal_test(manny["DO_mg_L"].values, period=12)
# Print easy-to-read output
print(f"Trend direction: {res_seasonal.trend}")
print(f"Significant trend (h): {res_seasonal.h}")
print(f"p-value: {res_seasonal.p:.4f}")
print(f"Kendall’s Tau: {res_seasonal.Tau:.3f}")
print(f"Sen’s slope: {res_seasonal.slope:.4f}")


In [ ]:
# If Autocorrelation present (common in env. series):
# This variant adjusts the test for serial correlation between consecutive readings, which otherwise
# can make the standard Mann-Kendall test overconfident (an artificially low p-value).
res_auto = mk.yue_wang_modification_test(manny["DO_mg_L"].values)

# Print easy-to-read output
print(f"Trend direction: {res_auto.trend}")
print(f"Significant trend (h): {res_auto.h}")
print(f"p-value: {res_auto.p:.4f}")
print(f"Kendall’s Tau: {res_auto.Tau:.3f}")
print(f"Sen’s slope: {res_auto.slope:.4f}")


## 3.8 Hierarchical Clustering on Water Quality Data
Clustering lets us ask a different kind of question: rather than testing a specific hypothesis, we let the data group itself into sites or readings that look similar across several variables at once, and then try to interpret what distinguishes each group. Hierarchical clustering is a natural choice here because it does not require us to decide the number of clusters in advance. We build the full hierarchy first, as a dendrogram, and only then decide how many clusters make sense for our purposes.


In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from scipy.cluster.hierarchy import linkage, dendrogram
import matplotlib.pyplot as plt

# Select numeric columns for clustering
numeric_cols = ['Salinity_ppt', 'DO_mg_L', 'pH', 'WaterTemp_C', 'Air_Temp_C']
wq_num = wq_drop[numeric_cols].dropna()   # drop rows with any missing numeric values

In [ ]:
wq_num

In [ ]:
## Standardise data using the standard scaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(wq_num)

In [ ]:
X_scaled

In [ ]:
linked = linkage(X_scaled, method='ward')  # 'ward' minimises variance within clusters
display(linked)

In [ ]:
# Plot dendrogram
plt.figure(figsize=(10, 5))
dendrogram(linked,
           orientation='top',
           distance_sort='descending',
           show_leaf_counts=False)
plt.title("Hierarchical Clustering Dendrogram – Water Quality")
plt.xlabel("Sample Index")
plt.ylabel("Euclidean Distance")
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

In [ ]:
## Extract and compare clusters
from scipy.cluster.hierarchy import fcluster

# Choose number of clusters (e.g. 3)
labels = fcluster(linked, t=7, criterion='maxclust')

# Add back to the dataframe
clustered = wq_num.copy()
clustered['Cluster'] = labels

# Summarise each cluster
cluster_summary = clustered.groupby('Cluster').mean().round(2)
print(cluster_summary)

In [ ]:
clustered

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Reuse the 'clustered' DataFrame from before
# (It includes the numeric features and the 'Cluster' label)

plt.figure(figsize=(8, 6))
sns.scatterplot(
    data=clustered,
    x="WaterTemp_C",
    y="DO_mg_L",
    hue="Cluster",
    palette="Set2",
    s=80,
    edgecolor="black",
    alpha=0.8
)

plt.title("Hierarchical Clusters in Water Quality Data")
plt.xlabel("Water Temperature (°C)")
plt.ylabel("Dissolved Oxygen (mg/L)")
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend(title="Cluster", loc="best")
plt.tight_layout()
plt.show()


---
# Part 4: Intro to **Spatial Data Visualisation**

Environmental data is very often spatial, and it is worth being able to plot it on a map rather than just as an abstract scatter. We will build a simple **GeoDataFrame** from the penguins data, which has had dummy longitude and latitude coordinates assigned to each island, and then add a **basemap** so the points sit in a recognisable geographic context. The coordinates here are illustrative rather than the real locations of these islands, but the workflow is exactly what you would use with real spatial data.


In [ ]:
import matplotlib.pyplot as plt
import geopandas as gpd
import seaborn as sns
import contextily as ctx

In [ ]:
# Minimal spatial demo (optional packages may be needed)
new_penguins = pd.read_csv("Penguins_with_coords.csv")
new_penguins

In [ ]:
gdf = gpd.GeoDataFrame(
    new_penguins,
    geometry=gpd.points_from_xy(new_penguins["lon"], new_penguins["lat"]),  # build point geometry from lon/lat columns
    crs="EPSG:4326"     # WGS84 coordinate reference system
)
gdf


In [ ]:
gdf.dtypes

In [ ]:
# Use Seaborn style for cleaner aesthetics
sns.set(style="whitegrid")

# Create figure and axis manually
fig, ax = plt.subplots(figsize=(8,6))

# Plot the GeoDataFrame with chosen colours and edge styles
gdf.plot(
    column="species",
    ax=ax,
    legend=True,
    markersize=60,
    alpha=0.9,
    edgecolor="black",
    cmap="Set2"   # Try 'Set1', 'Dark2', or 'tab10' for variety
)

# Add custom title and labels
ax.set_title("Penguin Species by Island", fontsize=14, fontweight="bold", pad=10)
ax.set_xlabel("Longitude (°)", fontsize=11)
ax.set_ylabel("Latitude (°)", fontsize=11)

# Optional: adjust legend position and style
leg = ax.get_legend()
leg.set_title("Species")
leg._legend_box.align = "left"  # left-align legend labels

# Add grid, frame, and aspect ratio for realism
ax.grid(True, linestyle="--", alpha=0.4)
ax.set_aspect("equal")   # preserve geographic proportions
ax.set_facecolor("#f6f6f6")

# Zoom into the penguin area automatically
ax.set_xlim(gdf.total_bounds[0]-0.2, gdf.total_bounds[2]+0.2)
ax.set_ylim(gdf.total_bounds[1]-0.2, gdf.total_bounds[3]+0.2)

# Clean up and show
sns.despine(left=False, bottom=False)
plt.tight_layout()
plt.show()

In [ ]:
# pip install contextily   # ← run once if not already installed
# Use a clean seaborn theme for better visuals
sns.set(style="whitegrid")

# Basemap tiles are served in the Web Mercator projection (EPSG:3857), so we reproject our
# WGS84 (lon/lat) points into that CRS first, otherwise the points and the tiles won't line up
gdf_web = gdf.to_crs(epsg=3857)

# Create the figure and axis manually (for full control)
fig, ax = plt.subplots(figsize=(6, 6))

# Plot the GeoDataFrame
gdf_web.plot(
    ax=ax,
    column="species",      # colour points by species
    legend=True,
    cmap="Set2",           # nice readable colour palette
    markersize=70,
    alpha=0.9,
    edgecolor="black"      # thin black border for visibility
)

# Add a light basemap (CartoDB Positron is good for clarity)
ctx.add_basemap(ax, source=ctx.providers.CartoDB.Positron)


# Adjust legend placement and style
leg = ax.get_legend()
leg.set_title("Species")
leg._legend_box.align = "left"

# Tidy up the axes and layout
ax.set_title("Penguin Species with Basemap", fontsize=14, fontweight="bold", pad=10)
ax.set_axis_off()
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
fig, axes = plt.subplots(1, 3, figsize=(6, 6))

for ax, provider, title in zip( #zip() combines two or more sequences (lists, tuples, or arrays) element-by-element
    axes,
    [ctx.providers.CartoDB.Positron, ctx.providers.NASAGIBS.BlueMarble, ctx.providers.Esri.WorldImagery],
    ["Positron (Clean)", "NASA", "Esri Satellite"]
):
    cmap = ListedColormap(["blue", "green", "red"])
    gdf_web.plot(ax=ax, column="species", cmap=cmap, markersize=50, alpha=0.9)
    # zoom sets the basemap tile resolution: higher values give sharper tiles but download more data
    ctx.add_basemap(ax, source=provider, zoom=8, crs=gdf_web.crs)
    ax.set_title(title)
    ax.set_axis_off()

plt.tight_layout()
plt.show()



---
## Practice Exercises
1. **Penguins:** Recreate violin plots for `bill_length_mm` by island and discuss any bimodality.  
2. **Air Quality:** Compute monthly means by location and plot a small multiple (FacetGrid) of months × locations.  
3. **Water Quality:** Try a **log-transform** on BOD or Nitrate, re-fit OLS & GLM, and compare residuals.  
4. **Clustering:** Increase to 4 clusters and profile each cluster’s mean and standard deviation across variables.  
5. **Spatial:** Add a new column for “region” (e.g., dummy values) and symbolise points by region + species.



### Notes & Caveats
- Always **document** preprocessing choices (drops, imputations, transforms, filters).
- Prefer **simple visuals** first, and only add complexity when it clarifies the story.
- Dual-axis plots can mislead: use them sparingly and label clearly.
- GLM/GAM choices depend on domain knowledge, so don't treat them as black boxes.
- For reproducibility, keep raw data read-only and export a **clean** version.
